In [9]:
from pyannote.audio import Model

model = Model.from_pretrained("pyannote/segmentation-3.0")
print(model)

PyanNet(
  (sincnet): SincNet(
    (wav_norm1d): InstanceNorm1d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
    (conv1d): ModuleList(
      (0): Encoder(
        (filterbank): ParamSincFB()
      )
      (1): Conv1d(80, 60, kernel_size=(5,), stride=(1,))
      (2): Conv1d(60, 60, kernel_size=(5,), stride=(1,))
    )
    (pool1d): ModuleList(
      (0-2): 3 x MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    )
    (norm1d): ModuleList(
      (0): InstanceNorm1d(80, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (1-2): 2 x InstanceNorm1d(60, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
    )
  )
  (lstm): LSTM(60, 128, num_layers=4, batch_first=True, dropout=0.5, bidirectional=True)
  (linear): ModuleList(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Linear(in_features=128, out_features=128, bias=True)
  )
  (classifier): Linear(in_features=128, out_features=7, bias=

In [ ]:
from pyannote.audio import Model, Pipeline
import torch

model = Model.from_pretrained("pyannote/segmentation-3.0")
model.to(torch.device('cuda'))


In [ ]:
from pyannote.audio import Model, Inference

model = Model.from_pretrained("pyannote/segmentation-3.0")  # или локальный путь
inference = Inference(model, step=0.25)  # шаг 0.25с обычно норм для границ

scores = inference("../../data/audio/audio-test.wav")  # SlidingWindowFeature
P = scores.data  # shape (T, C)
sw = scores.sliding_window

In [18]:
import numpy as np
from scipy.ndimage import uniform_filter1d

P_smooth = uniform_filter1d(P.astype(np.float32), size=5, axis=0)

delta = np.linalg.norm(P_smooth[1:] - P_smooth[:-1], axis=1)  # (T-1,)

# стартовый порог — верхний процентиль
thr = np.quantile(delta, 0.98)

idx = np.where(delta > thr)[0] + 1
change_times = [sw[i].middle for i in idx]
change_times[:20]

[np.float64(6.5),
 np.float64(7.75),
 np.float64(9.0),
 np.float64(9.25),
 np.float64(9.25),
 np.float64(9.5),
 np.float64(9.75),
 np.float64(68.25),
 np.float64(69.75),
 np.float64(69.75),
 np.float64(70.0),
 np.float64(70.0),
 np.float64(70.5),
 np.float64(70.5),
 np.float64(108.75),
 np.float64(108.75),
 np.float64(109.0),
 np.float64(109.0),
 np.float64(109.25),
 np.float64(109.25)]